# 🚀 ABSA Model Training - Shopping Support System

Notebook này huấn luyện 4 mô hình PhoBERT cho phân tích cảm xúc theo khía cạnh (ABSA):
- **Quality** (Chất lượng sản phẩm)
- **Price** (Giá cả)
- **Delivery** (Giao hàng)
- **Service** (Dịch vụ/CSKH)

## Hướng dẫn sử dụng:
1. Vào **Runtime → Change runtime type → GPU (T4)**
2. Chạy lần lượt từng ô (Cell) từ trên xuống dưới
3. Ở bước Upload Data, hãy tải lên 2 file CSV từ thư mục `data/`
4. Sau khi train xong, tải file `trained_models.zip` về máy và giải nén vào thư mục `models/`

## 1. Kiểm tra GPU và Cài đặt thư viện

In [ ]:
# Kiem tra GPU
!nvidia-smi
print("\n" + "="*50)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cai dat thu vien can thiet
!pip install -q transformers datasets scikit-learn accelerate

## 2. Upload dữ liệu

Chạy ô bên dưới, sau đó chọn 2 file từ thư mục `data/` trên máy:
- `auto_labeled_cleaned_positive_reviews.csv`
- `auto_labeled_cleaned_negative_reviews.csv`

In [ ]:
from google.colab import files
import pandas as pd

print("Hay chon 2 file CSV tu thu muc data/...")
uploaded = files.upload()

# Doc du lieu
df_pos = pd.read_csv('auto_labeled_cleaned_positive_reviews.csv', encoding='latin-1')
df_neg = pd.read_csv('auto_labeled_cleaned_negative_reviews.csv', encoding='latin-1')
df = pd.concat([df_pos, df_neg], ignore_index=True)

print(f"\nTong so du lieu: {len(df)} dong")
print(f"Cot: {list(df.columns)}")
print(f"\nPhan bo nhan:")
for col in ['Quality', 'Price', 'Delivery', 'Service']:
    print(f"  {col}: {dict(df[col].value_counts().sort_index())}")

## 3. Cấu hình & Hàm huấn luyện

In [ ]:
import torch
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

# ==========================================
# CAU HINH
# ==========================================
MODEL_NAME = "vinai/phobert-base-v2"
ASPECTS = ['Quality', 'Price', 'Delivery', 'Service']
LABEL_MAPPING = {-1: 0, 0: 1, 1: 2, 2: 3}
LABEL_NAMES = ['Khong nhac toi (-1)', 'Che (0)', 'Binh thuong (1)', 'Khen (2)']

# ==========================================
# DATASET CLASS
# ==========================================
class ShoppingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True,
            return_attention_mask=True, return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# COMPUTE METRICS
# ==========================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# Load Tokenizer (chi can 1 lan)
print(f"Dang tai Tokenizer {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer da san sang!")

In [ ]:
def train_aspect(aspect_name, df, tokenizer, epochs=3, batch_size=16):
    print(f"\n{'='*60}")
    print(f"  HUAN LUYEN MO HINH: {aspect_name.upper()}")
    print(f"{'='*60}")

    # Chuan bi du lieu
    temp_df = df[['cleaned_comment', aspect_name]].dropna()
    temp_df = temp_df[temp_df[aspect_name].isin(LABEL_MAPPING.keys())].copy()
    temp_df['label'] = temp_df[aspect_name].map(LABEL_MAPPING)

    print(f"  So mau: {len(temp_df)}")
    print(f"  Phan bo: {dict(temp_df[aspect_name].value_counts().sort_index())}")

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        temp_df['cleaned_comment'].astype(str).tolist(),
        temp_df['label'].tolist(),
        test_size=0.1,
        random_state=42,
        stratify=temp_df['label'].tolist()
    )

    train_dataset = ShoppingDataset(train_texts, train_labels, tokenizer)
    val_dataset = ShoppingDataset(val_texts, val_labels, tokenizer)

    output_dir = f'models/{aspect_name.lower()}_model'
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_steps=50,
        load_best_model_at_end=True,
        save_total_limit=1,
        fp16=True,
        report_to='none',
        dataloader_num_workers=2,
    )

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Danh gia cuoi cung
    eval_result = trainer.evaluate()
    print(f"\n  KET QUA {aspect_name.upper()}:")
    print(f"    Accuracy: {eval_result['eval_accuracy']:.4f}")
    print(f"    F1-Score: {eval_result['eval_f1']:.4f}")

    # Luu model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"  Da luu model tai: {output_dir}")

    return eval_result

## 4. Huấn luyện cả 4 mô hình

Chạy ô bên dưới để bắt đầu huấn luyện. Trên GPU T4, mỗi model mất khoảng **3-5 phút**.

In [ ]:
import time

all_results = {}
total_start = time.time()

for aspect in ASPECTS:
    start = time.time()
    result = train_aspect(aspect, df, tokenizer, epochs=3, batch_size=16)
    elapsed = time.time() - start
    all_results[aspect] = result
    print(f"  Thoi gian: {elapsed/60:.1f} phut")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"  TONG KET")
print(f"{'='*60}")
print(f"  Tong thoi gian: {total_time/60:.1f} phut")
for aspect, result in all_results.items():
    print(f"  {aspect:10s} | Acc: {result['eval_accuracy']:.4f} | F1: {result['eval_f1']:.4f}")

## 5. Tải models về máy

Chạy ô bên dưới để nén và tải toàn bộ 4 models về máy.

Sau khi tải xong, **giải nén `trained_models.zip`** rồi copy thư mục `models/` vào thư mục gốc của project:
```
Shopping_Support_System/
├── models/
│   ├── quality_model/
│   ├── price_model/
│   ├── delivery_model/
│   └── service_model/
├── src/
└── ...
```

In [ ]:
import shutil
from google.colab import files

# Xoa cac checkpoint trung gian de giam dung luong
for aspect in ASPECTS:
    model_dir = f'models/{aspect.lower()}_model'
    for item in os.listdir(model_dir):
        item_path = os.path.join(model_dir, item)
        if os.path.isdir(item_path) and 'checkpoint' in item:
            shutil.rmtree(item_path)
            print(f"  Da xoa checkpoint: {item_path}")

# Nen thanh file zip
shutil.make_archive('trained_models', 'zip', '.', 'models')
zip_size = os.path.getsize('trained_models.zip') / (1024*1024)
print(f"\nDa nen thanh cong! Dung luong: {zip_size:.1f} MB")

# Tai ve
print("Dang tai file ve may...")
files.download('trained_models.zip')